In [ ]:
import os, re, math
import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from peft import PeftModel


hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

BASE_MODEL = "meta-llama/Llama-3.2-3B"
HF_USER = "hoseingharakhani"
DATASET_NAME = f"{HF_USER}/used-car-price-cleand-prompts"


RUN_NAME = "price_2026-09-30_15.25.02_lite" 
HUB_MODEL_NAME = f"{HF_USER}/{RUN_NAME}"

In [ ]:
quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4"
)


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    HUB_MODEL_NAME,
    token=hf_token,
)

print("Model loaded successfully!")

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8, do_sample=False)

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0,prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

def evaluate(model_predict_func, test_data):
    results = []
    for item in tqdm(test_data):
        try:
            predicted_text = model_predict_func(item)

            match = re.search(r'(\d+(\.\d+)?)', predicted_text.replace(',', ''))
            predicted_price = float(match.group(1)) if match else 0.0

            actual_price = float(item['completion'].replace(',', '').strip())

            results.append({"actual": actual_price, "predicted": predicted_price})
        except Exception as e:

            results.append({"actual": float(item['comletion']), "predicted": 0.0})

    df = pd.DataFrame(results)
    df['error'] = df['actual'] - df['predicted']
    df['abs_error'] = abs(df['error'])
    df['percentage_error'] = df['error'] / df['actual'] * 100

    mae = df['abs_error'].mean()
    mape = df['percentage_error'].mean()

    print("\n" + "="*40)
    print("Mean Absolute Error (MAE): ${mea:,.2f}")
    print("Mean Absolute Percentage Error (MAPE): ${mape:,.2f}%")
    print("="*40 + "\n")


    plt.figure(figsize=(10, 6))
    plt.scatter(df['actual'], df['predicted'], alpha=0.4, color='blue')


    min_val = min(df['actual'].min(), df['predicted'].min())
    max_val = max(df['actual'].max(), df['predicted'].max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label="Perfect Prediction" )

    plt.xlabel('Actual Price($)')
    plt.ylabel('Predicted Price($)')
    plt.title('Actual vs Predicted Car Prices')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    return df
          


In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset["test"]

results_df = evaluate(model_predict, test)

print(results_df.head(10))

In [ ]:
mae = results_df['abs_error'].mean()
mape = results_df['percentage_error'].mean()

print("\n"+ "="*40)
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print("="*40)

mape_correct = (results_df['abs_error'] / results_df['actual'].replace(0, 1)).mean() * 100
print(f"📊 MAPE real: {mape_correct:.2f}%")